# Fetch data

In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features 
y = cirrhosis_patient_survival_prediction.data.targets 
y = y.iloc[:, 0]

# Prepare attributes, pipeline and split sets

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

# NaNN and NaN --> np.nan
X = X.replace(["NaN", "NaNN", "", " "], np.nan)

# Change categorical variables to numeric
cols_to_numeric = ["Cholesterol", "Copper", "Tryglicerides", "Platelets"]
X[cols_to_numeric] = X[cols_to_numeric].apply(pd.to_numeric, errors="coerce")

# Change Stage to category
X["Stage"] = X["Stage"].astype("category")

# Drop rows with missing attributes
X = X.dropna()
y = y.loc[X.index]

# Split data into training and test sets
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=67, stratify=y)

# Cross validation strategy
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=67)

# Preprocessor
cat_cols = X_rest.select_dtypes(include=["object", "str", "category"]).columns
num_cols = X_rest.select_dtypes(include=["number"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols,),
    ]
)

# # Preprocessor with imputation for future use
# preprocessor = ColumnTransformer(
#     transformers=[
#         ("num", SimpleImputer(strategy="median"), num_cols),
#         ("cat", Pipeline([
#             ("imputer", SimpleImputer(strategy="most_frequent")),
#             ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
#         ]), cat_cols),
#     ]
# )

# Bayes classificator

In [3]:

from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

model = Pipeline([
    ("prep", preprocessor),
    ("nb", GaussianNB())
])

scores = cross_val_score(model, X_rest, y_rest, cv=cv, scoring="accuracy")
print(f"Average accuracy scores for each CV fold:\n")
for i, score in enumerate(scores):
    print(f"\tFold {i+1}: {score:.4f}")
    
print(f"\nAverage Accuracy from CV: {scores.mean():.4f}\n")

model.fit(X_rest, y_rest)
y_pred = model.predict(X_test)

print("Test Set - Classification Report\n")
print(classification_report(y_test, y_pred))


Average accuracy scores for each CV fold:

	Fold 1: 0.6429
	Fold 2: 0.6429
	Fold 3: 0.5000
	Fold 4: 0.7500
	Fold 5: 0.7778
	Fold 6: 0.6296
	Fold 7: 0.5926
	Fold 8: 0.6667

Average Accuracy from CV: 0.6503

Test Set - Classification Report

              precision    recall  f1-score   support

           C       0.75      0.80      0.77        30
          CL       0.14      0.25      0.18         4
           D       0.65      0.50      0.56        22

    accuracy                           0.64        56
   macro avg       0.51      0.52      0.51        56
weighted avg       0.67      0.64      0.65        56



In [ ]:
from sklearn.tree import DecisionTreeClassifier


model = Pipeline(
    [
        ("prep", preprocessor),
        ("dt", DecisionTreeClassifier(random_state=67)),
    ]
)

scores = cross_val_score(model, X_rest, y_rest, cv=cv, scoring="accuracy")

print("Average accuracy scores for each CV fold:\n")
for i, score in enumerate(scores):
    print(f"\tFold {i+1}: {score:.4f}")

print(f"\nAverage Accuracy from CV: {scores.mean():.4f}\n")

model.fit(X_rest, y_rest)
y_pred = model.predict(X_test)

print("Test Set - Classification Report\n")
print(classification_report(y_test, y_pred))

Average accuracy scores for each CV fold:

	Fold 1: 0.7500
	Fold 2: 0.6429
	Fold 3: 0.6429
	Fold 4: 0.6429
	Fold 5: 0.7037
	Fold 6: 0.6296
	Fold 7: 0.7037
	Fold 8: 0.5926

Average Accuracy from CV: 0.6635

Test Set - Classification Report

              precision    recall  f1-score   support

           C       0.67      0.67      0.67        30
          CL       0.20      0.25      0.22         4
           D       0.62      0.59      0.60        22

    accuracy                           0.61        56
   macro avg       0.50      0.50      0.50        56
weighted avg       0.61      0.61      0.61        56



In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = Pipeline(
    [
        ("prep", preprocessor),
        ("rf", RandomForestClassifier(random_state=67)),
    ]
)

scores = cross_val_score(
    model_rf, X_rest, y_rest, cv=cv, scoring="accuracy"
)

print("Average accuracy scores for each CV fold:\n")
for i, score in enumerate(scores):
    print(f"\tFold {i+1}: {score:.4f}")

print(f"\nAverage Accuracy from CV: {scores.mean():.4f}\n")

model_rf.fit(X_rest, y_rest)
y_pred = model_rf.predict(X_test)

print("Test Set - Classification Report\n")
print(classification_report(y_test, y_pred, zero_division=0))

Average accuracy scores for each CV fold:

	Fold 1: 0.7500
	Fold 2: 0.7500
	Fold 3: 0.7143
	Fold 4: 0.7143
	Fold 5: 0.8519
	Fold 6: 0.7778
	Fold 7: 0.7037
	Fold 8: 0.7407

Average Accuracy from CV: 0.7503

Test Set - Classification Report

              precision    recall  f1-score   support

           C       0.80      0.80      0.80        30
          CL       0.00      0.00      0.00         4
           D       0.62      0.73      0.67        22

    accuracy                           0.71        56
   macro avg       0.47      0.51      0.49        56
weighted avg       0.67      0.71      0.69        56



In [7]:
from sklearn.svm import SVC

model_svm = Pipeline(
    [
        ("prep", preprocessor),
        ("svm", SVC(random_state=67)),
    ]
)

scores = cross_val_score(
    model_svm, X_rest, y_rest, cv=cv, scoring="accuracy"
)

print("Average accuracy scores for each CV fold:\n")
for i, score in enumerate(scores):
    print(f"\tFold {i+1}: {score:.4f}")

print(f"\nAverage Accuracy from CV: {scores.mean():.4f}\n")

model_svm.fit(X_rest, y_rest)
y_pred = model_svm.predict(X_test)

print("Test Set - Classification Report\n")
print(classification_report(y_test, y_pred, zero_division=0))

Average accuracy scores for each CV fold:

	Fold 1: 0.5357
	Fold 2: 0.5357
	Fold 3: 0.5357
	Fold 4: 0.6071
	Fold 5: 0.5185
	Fold 6: 0.5556
	Fold 7: 0.5556
	Fold 8: 0.4815

Average Accuracy from CV: 0.5407

Test Set - Classification Report

              precision    recall  f1-score   support

           C       0.55      0.97      0.70        30
          CL       0.00      0.00      0.00         4
           D       0.67      0.09      0.16        22

    accuracy                           0.55        56
   macro avg       0.40      0.35      0.29        56
weighted avg       0.56      0.55      0.44        56

